In [3]:
# !huggingface-cli login

!huggingface-cli whoami

Ap-star


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import BitsAndBytesConfig, LlamaTokenizer
from peft import PeftModel
from huggingface_hub import notebook_login

/mnt/c/Users/ogida/Desktop/Hope work/Tech/Vscode_files/transformers/LLM_finetunning/Verifi/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
if "COLAB_GPU" in os.environ:
  !huggingface-cli login
else:
  notebook_login()

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import torch




base_model_id = "meta-llama/Llama-3.2-1B"
adapter_path = "/models/Verifi-Adapter"
tokeniser_path = "/models/verifi-tokenizer"

# 1. FIX: Use AutoTokenizer and load your explicitly saved tokenizer with use_fast=True
tokenizer = AutoTokenizer.from_pretrained(tokeniser_path, use_fast=True)

# tokenizer = LlamaTokenizer.from_pretrained(base_model_id, use_fast=True, trust_remote_code=True, add_eos_token=True

                              #)

nf4Config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=nf4Config,
    device_map="auto",
    trust_remote_code=True,
    token=True
)

# Attach your fine-tuned Verifi weights
modelFinetuned = PeftModel.from_pretrained(base_model, adapter_path)
modelFinetuned.eval()


   

In [ ]:
user_question = "What is the difference between common stocks and preferred stocks?"

# 2. FIX: Use the exact Prompt Template from your training phase!
# Even if you don't have a 10-K context to provide right now, 
# you MUST use the structural tags so the model knows what to do.
eval_prompt = (
    f"### Financial Context:\nUse internal knowledge.\n\n"
    f"### Question:\n{user_question}\n\n"
    f"### Verified Answer:\n"
)

# Tokenize and generate
promptTokenized = tokenizer(eval_prompt, return_tensors="pt").to("cuda")



with torch.no_grad():
    outputs = modelFinetuned.generate(
        **promptTokenized, 
        max_new_tokens= 1024, # Reduced so it doesn't ramble endlessly
        pad_token_id=tokenizer.eos_token_id,
        use_cache=True # Turn cache back on for faster inference
    )


    generated_tokens = outputs
    # print(tokenizer.decode(modelFinetuned.generate(**promptTokenized, max_new_tokens=1024)[0], skip_special_tokens=True))
    response = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    
    print(response)

    torch.cuda.empty_cache()


